In [ ]:
import numpy as np
from PIL import Image

def rgb_to_ycbcr(image):
    """Преобразование RGB в YCbCr"""
    rgb = np.array(image, dtype=np.float32)
    
    r, g, b = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]

    y = 0.299 * r + 0.587 * g + 0.114 * b
    cb = 0.5 * r - 0.4187 * g - 0.0813 * b + 128
    cr = 0.1687 * r - 0.3313 * g + 0.5 * b + 128
    
    return y, cb, cr

def downsample_chroma(cb, cr):
    """Понижающая дискретизация цветовых компонент"""
    # Просто берем каждый второй пиксель по горизонтали и вертикали
    return cb[::2, ::2], cr[::2, ::2]

def dct_2d(block):
    """Дискретное косинусное преобразование для блока 8x8"""
    n = 8
    dct_block = np.zeros((n, n))
    
    for u in range(n):
        for v in range(n):
            sum_val = 0.0
            for i in range(n):
                for j in range(n):
                    cu = 1.0/np.sqrt(2) if u == 0 else 1.0
                    cv = 1.0/np.sqrt(2) if v == 0 else 1.0
                    cos_u = np.cos((2*i + 1) * u * np.pi / (2*n))
                    cos_v = np.cos((2*j + 1) * v * np.pi / (2*n))
                    sum_val += block[i,j] * cos_u * cos_v * cu * cv
            dct_block[u,v] = sum_val / 4
    
    return dct_block

def quantize(block, quant_matrix, component='Y'):
    """Квантование DCT коэффициентов"""
    if component == 'Y':
        # Матрица квантования для яркости
        q = np.array([
            [16, 11, 10, 16, 24, 40, 51, 61],
            [12, 12, 14, 19, 26, 58, 60, 55],
            [14, 13, 16, 24, 40, 57, 69, 56],
            [14, 17, 22, 29, 51, 87, 80, 62],
            [18, 22, 37, 56, 68, 109, 103, 77],
            [24, 35, 55, 64, 81, 104, 113, 92],
            [49, 64, 78, 87, 103, 121, 120, 101],
            [72, 92, 95, 98, 112, 100, 103, 99]
        ])
    else:
        # Матрица квантования для цветовых компонент
        q = np.array([
            [17, 18, 24, 47, 99, 99, 99, 99],
            [18, 21, 26, 66, 99, 99, 99, 99],
            [24, 26, 56, 99, 99, 99, 99, 99],
            [47, 66, 99, 99, 99, 99, 99, 99],
            [99, 99, 99, 99, 99, 99, 99, 99],
            [99, 99, 99, 99, 99, 99, 99, 99],
            [99, 99, 99, 99, 99, 99, 99, 99],
            [99, 99, 99, 99, 99, 99, 99, 99]
        ])
    
    if quant_matrix is not None:
        q = quant_matrix
    
    quantized = np.round(block / q)
    return quantized.astype(np.int32)

def zigzag_scan(block):
    """Зигзаг-сканирование для блоков 8x8"""
    result = np.zeros(64, dtype=block.dtype)
    index = 0
    
    for s in range(15): 
        if s < 8:
            if s % 2 == 0:
                for y in range(min(s, 7) + 1):
                    x = s - y
                    if x < 8 and y < 8:
                        result[index] = block[x, y]
                        index += 1
            else:
                for x in range(min(s, 7) + 1):
                    y = s - x
                    if x < 8 and y < 8:
                        result[index] = block[x, y]
                        index += 1
        else:
            if s % 2 == 0:
                for y in range(s - 7, 8):
                    x = s - y
                    if x < 8 and y < 8:
                        result[index] = block[x, y]
                        index += 1
            else:
                for x in range(s - 7, 8):
                    y = s - x
                    if x < 8 and y < 8:
                        result[index] = block[x, y]
                        index += 1
    
    return result

def rle_encode(arr):
    """RLE"""
    rle = []
    zero_count = 0
    
    for num in arr:
        if num == 0:
            zero_count += 1
        else:
            rle.append((zero_count, int(num)))
            zero_count = 0
    
    # Добавляем EOB (End Of Block)
    if zero_count > 0:
        rle.append((0, 0))
    
    return rle

def process_image(image_path, quant_matrix_Y=None, quant_matrix_C=None):
    """Основная функция обработки изображения"""
    # Загрузка изображения
    img = Image.open(image_path)
    
    # Шаг 1: Преобразование в YCbCr
    y, cb, cr = rgb_to_ycbcr(img)
    
    # Шаг 2: Понижающая дискретизация цветовых компонент
    cb_down, cr_down = downsample_chroma(cb, cr)
    
    # Подготовка данных для всех компонент
    components = {
        'Y': (y, quant_matrix_Y),
        'Cb': (cb_down, quant_matrix_C),
        'Cr': (cr_down, quant_matrix_C)
    }
    
    compressed_data = {}
    
    for name, (data, q_matrix) in components.items():
        height, width = data.shape
        blocks = []
        
        # Обработка каждого блока 8x8
        for i in range(0, height, 8):
            for j in range(0, width, 8):
                block = data[i:i+8, j:j+8]
                if block.shape != (8, 8):
                    # Дополнение последних блоков если нужно
                    padded = np.zeros((8, 8))
                    padded[:block.shape[0], :block.shape[1]] = block
                    block = padded
                
                # Шаг 3: Применение DCT
                dct_block = dct_2d(block)
                
                # Шаг 4: Квантование
                quant_block = quantize(dct_block, q_matrix, name)
                
                # Шаг 5: Зигзаг-сканирование
                zigzag = zigzag_scan(quant_block)
                
                # Шаг 6: RLE кодирование
                rle = rle_encode(zigzag)
                
                blocks.append(rle)
        
        compressed_data[name] = blocks
    
    return compressed_data

In [59]:
# Обработка изображения
compressed = process_image("origins/friren.jpg")

In [64]:
# Вывод информации о сжатых данных
print("Сжатые данные:")
for component in ['Y', 'Cb', 'Cr']:
    print(f"\nКомпонента {component}:")
    for i, block in enumerate(compressed[component][200:220]):
        print(f"Блок {i+1}: {block}")

Сжатые данные:

Компонента Y:
Блок 1: [(0, 112), (1, -4), (0, -1), (0, 1), (0, -1), (3, -1), (0, 0)]
Блок 2: [(0, 111), (0, 1), (0, -4), (0, 0)]
Блок 3: [(0, 110), (0, 1), (0, -5), (0, 1), (4, -1), (0, 0)]
Блок 4: [(0, 115), (0, -3), (0, -7), (2, -1), (0, -1), (0, 1), (0, 1), (3, 1), (0, 0)]
Блок 5: [(0, 119), (1, -2), (0, -1), (0, 0)]
Блок 6: [(0, 116), (0, 6), (0, -5), (0, -1), (0, 2), (0, -2), (2, -1), (3, 1), (0, 0)]
Блок 7: [(0, 110), (0, -1), (0, -3), (0, 0)]
Блок 8: [(0, 109), (0, 1), (0, -4), (0, 1), (0, -1), (0, -1), (3, -1), (0, 0)]
Блок 9: [(0, 105), (1, -1), (0, 0)]
Блок 10: [(0, 104), (1, -2), (2, 1), (0, 0)]
Блок 11: [(0, 104), (1, -1), (0, 0)]
Блок 12: [(0, 104), (0, -1), (0, -1), (0, 0)]
Блок 13: [(0, 107), (1, -1), (0, 0)]
Блок 14: [(0, 109), (0, -2), (0, -1), (1, -1), (0, 1), (0, 0)]
Блок 15: [(0, 114), (0, -1), (0, 1), (0, -1), (0, 0)]
Блок 16: [(0, 114), (0, 2), (0, 2), (2, -1), (0, 0)]
Блок 17: [(0, 108), (0, 2), (0, -1), (1, 1), (0, 1), (0, 0)]
Блок 18: [(0, 26), 

In [61]:
# Создаём тестовую матрицу 8x8
matrix = np.arange(64).reshape(8, 8)
print("Исходная матрица:\n", matrix)

# Применяем зигзаг-сканирование
zigzag_vector = zigzag_scan(matrix)
print("\nЗигзаг-вектор:\n", zigzag_vector)

Исходная матрица:
 [[ 0  1  2  3  4  5  6  7]
 [ 8  9 10 11 12 13 14 15]
 [16 17 18 19 20 21 22 23]
 [24 25 26 27 28 29 30 31]
 [32 33 34 35 36 37 38 39]
 [40 41 42 43 44 45 46 47]
 [48 49 50 51 52 53 54 55]
 [56 57 58 59 60 61 62 63]]

Зигзаг-вектор:
 [ 0  1  8 16  9  2  3 10 17 24 32 25 18 11  4  5 12 19 26 33 40 48 41 34
 27 20 13  6  7 14 21 28 35 42 49 56 57 50 43 36 29 22 15 23 30 37 44 51
 58 59 52 45 38 31 39 46 53 60 61 54 47 55 62 63]
